In [1]:
import pandas as pd
import torch as t
from torch_geometric.loader import DataLoader

import warnings
warnings.filterwarnings("ignore")

from data.featurize import SolvationDataset
from model.mpnn import MPNNModel
from model.gcn import GCNModel
from model.gat import GATModel

In [2]:
def predict_all_models(test_csv, mpnn_model_path, gcn_model_path, gat_model_path):
    """
    Make predictions on the test set using base line models.
    
    Returns a dataframe with true and predicted values.
    """
    test_dataset = SolvationDataset(test_csv)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    sample = test_dataset[0]
    node_feature_dim = sample.x.size(1)
    edge_feature_dim = sample.edge_attr.size(1) if sample.edge_attr.size(0) > 0 else 4

    device = t.device("cuda" if t.cuda.is_available() else "cpu")

    mpnn_model = MPNNModel(
        node_feature_dim, edge_feature_dim, hidden_dim=48, num_layers=2
    ).to(device)
    mpnn_model.load_state_dict(t.load(mpnn_model_path, map_location=device))
    mpnn_model.eval()

    gcn_model = GCNModel(node_feature_dim, hidden_dim=64, num_layers=3).to(device)
    gcn_model.load_state_dict(t.load(gcn_model_path, map_location=device))
    gcn_model.eval()

    gat_model = GATModel(node_feature_dim, hidden_dim=64, num_layers=3, heads=4).to(
        device
    )
    gat_model.load_state_dict(t.load(gat_model_path, map_location=device))
    gat_model.eval()

    results = []
    with t.no_grad():
        for data in test_loader:
            data = data.to(device)
            true_y = data.y.item()
            mpnn_pred = mpnn_model(data).item()
            gcn_pred = gcn_model(data).item()
            gat_pred = gat_model(data).item()
            results.append(
                {
                    "True": true_y,
                    "MPNN_Pred": mpnn_pred,
                    "GCN_Pred": gcn_pred,
                    "GAT_Pred": gat_pred,
                }
            )

    df_results = pd.DataFrame(results)

    return df_results

In [5]:
df_preds = predict_all_models(test_csv="data/CombiSolv-QM-sample-1.csv",
    mpnn_model_path="checkpoints/mpnn_solvation_model.pt",
    gcn_model_path="checkpoints/gcn_solvation_model.pt",
    gat_model_path="checkpoints/gat_solvation_model.pt",)
display(df_preds.head())

,True,MPNN_Pred,GCN_Pred,GAT_Pred
0,-7.955403,NaN,-4.613610,-6.441754
1,-2.673888,NaN,-4.302685,-6.212682
2,-13.015916,NaN,-4.228832,-6.111909
3,-12.029929,NaN,-4.644444,-6.672758
4,-4.570783,NaN,-4.755408,-6.585802
